In [ ]:
#Se instala las librerías necesarias
%pip install selenium pandas openpyxl webdriver-manager

## ⚙️ Step 1 — Install Dependencies

We install the four required libraries. Using `%pip` (instead of `!pip`) ensures the packages are installed in the same Python environment that the notebook kernel is using.

# 🎓 Task 1 — Web Scraping: UNMSM Admissions 2026-II

This notebook automates the extraction of applicant results from the official UNMSM (Universidad Nacional Mayor de San Marcos) admissions website for the **2026-II cycle**.

Using **Selenium**, the scraper navigates the site, collects all 111 academic programs, paginates through every result page, and consolidates everything into a single Excel file with **25,880 total records**.

**Target:** `https://admision.unmsm.edu.pe/Website20262/A/A.html`

---

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# 1. Se configura  el servicio para que descargue el driver automáticamente
service = Service(ChromeDriverManager().install())

# 2. Se inicia el navegador (Chrome)
driver = webdriver.Chrome(service=service)

# 3. Se maximiza la ventana para que Selenium "vea" todos los elementos
driver.maximize_window()

## 🌐 Step 2 — Launch the Browser

We use `webdriver-manager` to automatically download the correct version of ChromeDriver — no manual setup needed. The browser window is maximized so Selenium can detect all elements correctly, even those that only render at full screen width.

In [3]:
# 1. Se importa lo necesario para las esperas
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 2. Se define la URL de inicio (Asegúrate de que el driver esté abierto)
url_principal = "https://admision.unmsm.edu.pe/Website20262/A/A.html"
driver.get(url_principal)

# 3. Espera explícita: Aguardamos a que la tabla cargue en el DOM
wait = WebDriverWait(driver, 20)
# Se usa el selector que refleja la jerarquía que se encontró:
# table.table-striped busca la tabla, y el espacio + 'a' busca los links dentro
selector_carreras = "table.table-striped tbody tr td a"
enlaces_carreras = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, selector_carreras)))

# 4. Se guarda la información en una lista de diccionarios
lista_carreras = []
for e in enlaces_carreras:
    lista_carreras.append({
        "carrera": e.text.strip(),
        "url": e.get_attribute("href")  # Esto extrae el link del atributo href
    })

# 5. Se muestra la cantidad de carreras encontradas
print(f"Se han detectado {len(lista_carreras)} carreras.")

# 6. Le echamos un ojito al diccionario que contiene a las carreras con sus respectivo URL
lista_carreras

# Con este código se puede comprobar si la extracción fue correcta
# var_name= [c for c in lista_carreras if "Aquí pon el nombre de la carrera" in c['carrera']]
#print(f"La carrera {var_name}" fue encontrada.)

Se han detectado 111 carreras.


[{'carrera': 'ADMINISTRACIÓN',
  'url': 'https://admision.unmsm.edu.pe/Website20262/A/091/results.html'},
 {'carrera': 'ADMINISTRACIÓN - CHILCA',
  'url': 'https://admision.unmsm.edu.pe/Website20262/A/0914/results.html'},
 {'carrera': 'ADMINISTRACIÓN - HUARAL',
  'url': 'https://admision.unmsm.edu.pe/Website20262/A/0912/results.html'},
 {'carrera': 'ADMINISTRACIÓN - S.J.L',
  'url': 'https://admision.unmsm.edu.pe/Website20262/A/0911/results.html'},
 {'carrera': 'ADMINISTRACIÓN - VILLA RICA',
  'url': 'https://admision.unmsm.edu.pe/Website20262/A/0915/results.html'},
 {'carrera': 'ADMINISTRACIÓN DE LA GASTRONOMÍA',
  'url': 'https://admision.unmsm.edu.pe/Website20262/A/094/results.html'},
 {'carrera': 'ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES',
  'url': 'https://admision.unmsm.edu.pe/Website20262/A/093/results.html'},
 {'carrera': 'ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES - HUARAL',
  'url': 'https://admision.unmsm.edu.pe/Website20262/A/0932/results.html'},
 {'carrera': 'ADMINISTRAC

## 🗂️ Step 3 — Collect All Program URLs

We navigate to the main admissions page and extract the name and URL of every academic program listed in the results table. An explicit wait (`WebDriverWait`) ensures we only proceed once the table has fully loaded in the DOM — this avoids empty results caused by slow page rendering.

> **Result:** 111 academic programs detected across all faculties of UNMSM.

In [4]:
import time
import pandas as pd
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC

def extraer_postulantes_carrera(driver, url_carrera, nombre_carrera):
    """
    Navega por la carrera, cambia a 100 registros y usa el botón 'Siguiente'
    hasta que se deshabilite, capturando datos de atributos data-score/merit.
    """
    driver.get(url_carrera)
    wait = WebDriverWait(driver, 25) # Espera robusta para San Marcos
    
    # 1. Intentar poner la visualización en 100 registros (ID: dt-length-0)
    try:
        select_element = wait.until(EC.element_to_be_clickable((By.ID, "dt-length-0")))
        select = Select(select_element)
        select.select_by_value("100")
        time.sleep(5) # Tiempo para que DataTables procese el cambio
    except:
        print(f"No se pudo cambiar a 100 registros en {nombre_carrera}")

    resultados_carrera = []
    
    while True:
        # 2. Esperar a que la tabla de la HOJA ACTUAL cargue
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#tablaPostulantes tbody tr")))
        filas = driver.find_elements(By.CSS_SELECTOR, "#tablaPostulantes tbody tr")
        
        # 3. Extraer datos de la hoja en la que estamos
        for fila in filas:
            columnas = fila.find_elements(By.TAG_NAME, "td")
            if len(columnas) >= 6:
                # SOLUCIÓN DATOS VACÍOS: Extraemos de los atributos del HTML
                puntaje = columnas[3].get_attribute("data-score")
                merito = columnas[4].get_attribute("data-merit")
                
                resultados_carrera.append({
                    "Carrera": nombre_carrera,
                    "Código": columnas[0].text.strip(),
                    "Apellidos y Nombres": columnas[1].text.strip(),
                    "Escuela": columnas[2].text.strip(),
                    "Puntaje": puntaje if puntaje else "0.000",
                    "Mérito E.P.": merito if merito else "",
                    "Observación": columnas[5].text.strip()
                })

        # 4. LÓGICA DE MOVIMIENTO: Clic en 'Siguiente' (Paginación)
        try:
            # Identificamos el li que envuelve al botón Siguiente
            # El selector li.dt-paging-button:has(button.next) es el más preciso para tu HTML
            li_siguiente = driver.find_element(By.CSS_SELECTOR, "li.dt-paging-button:has(button.next)")
            
            # Si el contenedor li tiene la clase 'disabled', paramos el bucle
            if "disabled" in li_siguiente.get_attribute("class"):
                print(f"Fin de {nombre_carrera} alcanzado. Registros: {len(resultados_carrera)}")
                break
            
            # Si no está disabled, buscamos el botón interno y clickeamos con JS
            boton_next = li_siguiente.find_element(By.CSS_SELECTOR, "button.next")
            driver.execute_script("arguments[0].click();", boton_next)
            
            # ESPERA CRUCIAL: Pausa para que el contenido de la tabla cambie realmente
            time.sleep(4) 
            
        except Exception as e:
            print(f"Terminando paginación por seguridad en {nombre_carrera}")
            break
            
    return resultados_carrera

## 🔧 Step 4 — Define the Scraping Function

We define a reusable function `extraer_postulantes_carrera()` that handles one academic program at a time. It navigates to the program's results page, switches the table display to 100 rows per page, and iterates through every page using the "Next" button until it is disabled.

Key design decisions:
- **`data-score` and `data-merit` attributes** — the score and merit values are embedded in HTML attributes, not visible text. Direct `element.text` would return empty strings, so we use `get_attribute()` instead.
- **JavaScript click** — the pagination button is clicked via `execute_script()` to bypass potential overlay issues that block standard Selenium clicks.
- **`time.sleep(4)`** — a deliberate pause after each page turn gives DataTables time to re-render the new rows before we read them.

In [5]:
import os

# 1. Aseguramos que la lista 'todos_los_datos' empiece limpia
todos_los_datos = []

# 2. BUCLE MAESTRO: Iteramos por TODAS las carreras de 'lista_carreras'
# (Eliminamos el [:2] para que recorra la lista completa)
for i, carrera_info in enumerate(lista_carreras):
    print(f"[{i+1}/{len(lista_carreras)}] Procesando: {carrera_info['carrera']}")
    
    try:
        # Llamamos a tu función corregida
        datos = extraer_postulantes_carrera(driver, carrera_info['url'], carrera_info['carrera'])
        
        # Consolidamos los datos encontrados
        todos_los_datos.extend(datos)
        print(f"Éxito: {len(datos)} registros capturados.")
        
    except Exception as e:
        print(f"Error en {carrera_info['carrera']}: {e}")
        # Si falla una, continuamos con la siguiente para no perder el progreso
        continue

# 3. CONSOLIDACIÓN FINAL EN PANDAS
if todos_los_datos:
    df_final = pd.DataFrame(todos_los_datos)
    
    # 4. EXPORTACIÓN A EXCEL (Carpeta output/)
    output_path = "output/resultados_sanmarcos.xlsx"
    
    # Usamos index=False para que no cree la columna de números a la izquierda
    df_final.to_excel(output_path, index=False)
    
    print("-" * 30)
    print(f"¡PROCESO COMPLETADO!")
    print(f"Total de registros consolidados: {len(df_final)}")
    print(f"Archivo guardado en: {output_path}")
else:
    print("No se recolectó ninguna información. Revisa el driver.")

# 5. Cerramos el navegador al terminar todo
driver.quit()

[1/111] Procesando: ADMINISTRACIÓN
Fin de ADMINISTRACIÓN alcanzado. Registros: 553
Éxito: 553 registros capturados.
[2/111] Procesando: ADMINISTRACIÓN - CHILCA
Fin de ADMINISTRACIÓN - CHILCA alcanzado. Registros: 33
Éxito: 33 registros capturados.
[3/111] Procesando: ADMINISTRACIÓN - HUARAL
Fin de ADMINISTRACIÓN - HUARAL alcanzado. Registros: 31
Éxito: 31 registros capturados.
[4/111] Procesando: ADMINISTRACIÓN - S.J.L
Fin de ADMINISTRACIÓN - S.J.L alcanzado. Registros: 44
Éxito: 44 registros capturados.
[5/111] Procesando: ADMINISTRACIÓN - VILLA RICA
Fin de ADMINISTRACIÓN - VILLA RICA alcanzado. Registros: 33
Éxito: 33 registros capturados.
[6/111] Procesando: ADMINISTRACIÓN DE LA GASTRONOMÍA
Fin de ADMINISTRACIÓN DE LA GASTRONOMÍA alcanzado. Registros: 127
Éxito: 127 registros capturados.
[7/111] Procesando: ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES
Fin de ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES alcanzado. Registros: 805
Éxito: 805 registros capturados.
[8/111] Procesando: ADMINI

---
## 📊 Step 6 — Data Analysis

Now that all the data is consolidated in `df_final`, we explore it to extract meaningful insights: which programs received the most applicants, how many candidates were admitted vs eliminated, and what the top scores look like across all programs.

In [13]:
import pandas as pd

# Cargamos el Excel ya generado para el análisis
df_final = pd.read_excel("output/resultados_sanmarcos.xlsx")

# Resumen general
total_postulantes = len(df_final)
total_carreras    = df_final["Carrera"].nunique()

print("📋 General Summary — UNMSM Admissions 2026-II")
print(f"  Total applicant records : {total_postulantes:,}")
print(f"  Total academic programs : {total_carreras}")
print(f"  Columns available       : {list(df_final.columns)}")
print()
print(df_final.head())

📋 General Summary — UNMSM Admissions 2026-II
  Total applicant records : 25,880
  Total academic programs : 111
  Columns available       : ['Carrera', 'Código', 'Apellidos y Nombres', 'Escuela', 'Puntaje', 'Mérito E.P.', 'Observación']

          Carrera  Código               Apellidos y Nombres         Escuela  \
0  ADMINISTRACIÓN  656550      ABAD SALGADO, SOFIA ESTRELLA  ADMINISTRACIÓN   
1  ADMINISTRACIÓN  565808   ACUÑA ECHEBARRIA, YAMILA SORAYA  ADMINISTRACIÓN   
2  ADMINISTRACIÓN  658780      ACUÑA PACOTAIPE, DULCE MARIA  ADMINISTRACIÓN   
3  ADMINISTRACIÓN  651014  AGUERO VIDARTE, ADRIAN SEBASTIAN  ADMINISTRACIÓN   
4  ADMINISTRACIÓN  658610       AGUIRRE RAMOS, JOGAN ALYAIR  ADMINISTRACIÓN   

    Puntaje  Mérito E.P.                                       Observación  
0   461.375          NaN                                               NaN  
1  1074.875         41.0                                   ALCANZÓ VACANTE  
2   797.375          NaN                                

### 🏆 Top 10 Most Competitive Programs

Which programs attracted the most applicants? High applicant counts usually indicate high demand and, consequently, tougher competition for available spots.

In [12]:
# Top 10 carreras por número de postulantes
top10_carreras = (
    df_final.groupby("Carrera")
    .size()
    .reset_index(name="total_postulantes")
    .sort_values("total_postulantes", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

print("🏆 Top 10 Most Competitive Programs (by number of applicants):")
display(top10_carreras)

🏆 Top 10 Most Competitive Programs (by number of applicants):


,Carrera,total_postulantes
0,MEDICINA HUMANA,4350
1,DERECHO,1873
2,PSICOLOGÍA,1110
3,INGENIERÍA DE SISTEMAS,875
4,INGENIERÍA INDUSTRIAL,866
5,ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,805
6,ECONOMÍA,659
7,INGENIERÍA CIVIL,642
8,ENFERMERÍA,602
9,ADMINISTRACIÓN,553


### ✅ Admission Results — Who Got In?

The `Observación` column contains the admission status of each applicant. We analyze how many were admitted (`ingresante`) versus eliminated across all programs.

In [11]:
# Primero vemos los valores reales en la columna Observación
print("📋 Unique values in 'Observación' column:")
print(df_final["Observación"].value_counts().to_string())

# Filtramos ingresantes usando los valores exactos que aparecen
ingresantes = df_final[df_final["Observación"].str.upper().str.contains("INGRESANTE", na=False)]

obs_counts = (
    df_final["Observación"]
    .value_counts()
    .reset_index()
)
obs_counts.columns = ["observacion", "total"]
obs_counts["porcentaje"] = (obs_counts["total"] / len(df_final) * 100).round(2)

print("\n📊 Admission Results Distribution:")
display(obs_counts)

📋 Unique values in 'Observación' column:
Observación
ALCANZÓ VACANTE                                               4764
Articulo N° 5 del Reglamento de Admisión 2026-II               323
AUSENTE                                                        247
AUSENTE - Articulo N° 5 del Reglamento de Admisión 2026-II       3

📊 Admission Results Distribution:


,observacion,total,porcentaje
0,ALCANZÓ VACANTE,4764,18.41
1,Articulo N° 5 del Reglamento de Admisión 2026-II,323,1.25
2,AUSENTE,247,0.95
3,AUSENTE - Articulo N° 5 del Reglamento de Admi...,3,0.01


### 🥇 Top Scorers Across All Programs

Who achieved the highest admission scores in the entire exam? We convert the `Puntaje` column to numeric and rank applicants across all programs.

In [9]:
# Convertimos Puntaje a numérico (algunos valores pueden ser strings)
df_final["Puntaje"] = pd.to_numeric(df_final["Puntaje"], errors="coerce")

# Top 10 puntajes más altos
top_scores = (
    df_final[["Apellidos y Nombres", "Carrera", "Puntaje", "Observación"]]
    .dropna(subset=["Puntaje"])
    .sort_values("Puntaje", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

print("🥇 Top 10 Highest Scores Across All Programs:")
display(top_scores)

🥇 Top 10 Highest Scores Across All Programs:


,Apellidos y Nombres,Carrera,Puntaje,Observación
0,"VILCA CAQUI, ANGEL HANS",INGENIERÍA BIOMÉDICA,1717.375,ALCANZÓ VACANTE
1,"ESPINOZA ROJAS, GINO DANIEL",DERECHO,1638.000,ALCANZÓ VACANTE
2,"MAURICIO SOLORZANO, CRISTHIAN PIERO",MEDICINA HUMANA,1624.875,ALCANZÓ VACANTE
3,"SANCHEZ AZAÑA, JHANPIER ANGEL",MEDICINA HUMANA,1620.875,ALCANZÓ VACANTE
4,"BRUNO ANCHAISE, GIOVANNI PAOLO",INGENIERÍA INDUSTRIAL,1613.375,ALCANZÓ VACANTE
5,"FLORES JUNCO, GABRIEL ALONSO",MEDICINA HUMANA,1565.500,ALCANZÓ VACANTE
6,"LAZO TOVAR, MAXIMO VALENTINO",ARQUITECTURA Y URBANISMO,1565.500,ALCANZÓ VACANTE
7,"CASTRO SUCLUPE, KEVIN DAVID",INGENIERÍA DE SISTEMAS,1557.500,ALCANZÓ VACANTE
8,"HEREDIA MENDOZA, ALEXANDER PAOLO VICTOR",ECONOMÍA,1545.500,ALCANZÓ VACANTE
9,"FLORES ALARCON, JORGE ALEJANDRO",MATEMÁTICA,1537.500,ALCANZÓ VACANTE


### 📉 Least Demanded Programs

On the other end, which programs had the fewest applicants? This can reflect niche careers, less public awareness, or very specific admission profiles.

In [8]:
# Bottom 10 carreras por número de postulantes
bottom10_carreras = (
    df_final.groupby("Carrera")
    .size()
    .reset_index(name="total_postulantes")
    .sort_values("total_postulantes", ascending=True)
    .head(10)
    .reset_index(drop=True)
)

print("📉 Top 10 Least Demanded Programs (by number of applicants):")
display(bottom10_carreras)

📉 Top 10 Least Demanded Programs (by number of applicants):


,Carrera,total_postulantes
0,DANZA,6
1,INGENIERÍA ELÉCTRICA - CHILCA,18
2,CONTABILIDAD - VILLA RICA,19
3,CONTABILIDAD - HUARAL,20
4,EDUCACIÓN FÍSICA - OYÓN,20
5,ADMINISTRACION MARITIMA Y PORTUARIA - CHANCAY,21
6,CONTABILIDAD - OYÓN,21
7,INGENIERÍA GEOLÓGICA - OYÓN,23
8,ADMINISTRACIÓN DE TURISMO - S.J.L,25
9,INVESTIGACIÓN OPERATIVA - CHILCA,27


### 🎯 Average Score per Program (Top 15)

Which programs have the highest average admission scores? A high average suggests that the admitted pool was very competitive — the cutoff bar was raised by the strength of the applicant field.

In [7]:
# Promedio de puntaje por carrera — solo carreras con al menos 50 postulantes
# (evitamos distorsiones por muestras pequeñas como Danza o carreras de sede)
avg_score_by_carrera = (
    df_final.groupby("Carrera")
    .agg(total_postulantes=("Puntaje", "count"), avg_puntaje=("Puntaje", "mean"))
    .reset_index()
    .query("total_postulantes >= 50")  # mínimo 50 registros para que el promedio sea representativo
    .sort_values("avg_puntaje", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
avg_score_by_carrera["avg_puntaje"] = avg_score_by_carrera["avg_puntaje"].round(3)

print("🎯 Top 15 Programs by Average Admission Score (min. 50 applicants):")
display(avg_score_by_carrera)

🎯 Top 15 Programs by Average Admission Score (min. 50 applicants):


,Carrera,total_postulantes,avg_puntaje
0,CIENCIA DE LA COMPUTACIÓN,134,973.630
1,INGENIERÍA BIOMÉDICA,239,966.876
2,INGENIERÍA DE TELECOMUNICACIONES,125,946.820
3,EDUCACIÓN FÍSICA,160,928.912
4,HISTORIA,80,918.797
5,INGENIERÍA ELECTRÓNICA,255,909.634
6,INGENIERÍA MECATRONICA,291,902.711
7,LINGUÍSTICA,62,901.298
8,INGENIERÍA QUÍMICA,173,899.626
9,INGENIERÍA CIVIL,642,898.677


## 🚀 Step 5 — Run the Full Scraping Loop

We iterate over all 111 programs, call the scraping function for each one, and consolidate every record into a single DataFrame. If a program fails for any reason, the `try/except` block logs the error and continues to the next one — so a single broken page does not stop the entire process.

At the end, the consolidated DataFrame is exported to `output/resultados_sanmarcos.xlsx`.

> **Final result:** 25,880 applicant records across 111 programs.